## Transformations applied to Races Dataset 


The notebook below did the following:

1 Read races Table from Bronze layer

2 Only required columns for analytics were retailned (Excluded URL column)

3 Standardized column names using snake_case (raceName, circuitId)

4 Renamed columns

5 removed duplicates

6 Transformed columns to  title case (race_name)

7 Wrote transformed data into the silver layer- races table

In [0]:
%run "../00-Common/01. Env Config"

In [0]:
# Read & Write tables for Races

bronze_table=f"{catalog_name}.{bronze_schema}.races"
silver_table=f"{catalog_name}.{silver_schema}.races"

In [0]:
races_df= spark.read.table(bronze_table)

In [0]:
#Keeping Relevant tables
races_df=races_df.select(
    F.col("season"),
    F.col("round"),
    #F.col("url"),
    F.col("racesName"),
    F.col("date").alias("race_date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"),
    F.col("source_file")

)

In [0]:
# Standardizing the columns

races_df=( races_df.withColumnsRenamed(
                                {'racesName':'race_name',
                                'circuitid':'circuit_id'}
))

In [0]:
# finding Duplicates

'''
dupes= (races_df.groupBy(races_df.columns)
.count()
.filter(F.col("count")>1))
dupes.show()
'''


In [0]:
#Removing Duplicates

races_df_deduped= races_df.dropDuplicates()

In [0]:
#Standardizing Column naming convention
races_final_df=(races_df_deduped
                                .withColumn("race_name",F.initcap(F.col("race_name")))    
                                .withColumn("circuit_id",F.initcap(F.col("circuit_id")))
                )

In [0]:
#Writing Final table into Silver Layer

(

races_final_df
.write
.format("delta")
.mode("overwrite")
.saveAsTable(silver_table)
)


In [0]:
display(races_final_df)